<a href="https://colab.research.google.com/github/simoambr/Alpaca_Trading/blob/main/Alpaca_Run_Comparison_Shrinking_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# %% [CELL 1] ================================================================
# SETUP — installs, imports, config. Run once.
# =============================================================================
import subprocess, sys
for pkg in ("statsmodels", "pyarrow"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import glob, io, os, re, time, zipfile
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ---- Google Drive (Colab only; no-op elsewhere) ---------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass  # not running in Colab -- assume paths below are already reachable

# ---- paths ------------------------------------------------------------
# BASE_DIR is the parent folder in your Drive. SOURCE_FOLDERS is the list of
# sub-folders (each expected to contain .zip files, non-recursive) to pull
# data from -- add or remove folder names here to include/exclude a run
# without touching anything else. Order doesn't matter.
BASE_DIR = "/content/drive/MyDrive/Colab/Alpaca_Experiment_Runs"
SOURCE_FOLDERS = [
    "RUNS_Orb_15_30_Multiparameters/2023",
    "RUNS_Orb_15_30_Multiparameters/2024",
    "RUNS_Orb_15_30_Multiparameters/2025",
    "RUNS_Orb_10_20_MultiParameters/2023",
    "RUNS_Orb_10_20_MultiParameters/2024",
    "RUNS_Orb_10_20_MultiParameters/2025",
    "RUNS_Orb_10_20_MultiParameters/2024_V2",
    "RUNS_Orb_10_20_MultiParameters/2025_V2",
    "RUNS_Orb5_MultiParameters/2023",
    "RUNS_Orb5_MultiParameters/2024",
    "RUNS_Orb5_MultiParameters/2025",
    "RUNS_Orb5_MultiParameters/2026",
    # add more experiment/year folders here as they're produced
]

MASTER_DIR = "/content/master_cache"       # compact Parquet "source of truth"
OUT_XLSX = "/content/combo_analysis.xlsx"  # the human-facing analysis workbook
os.makedirs(MASTER_DIR, exist_ok=True)


def gather_zip_paths(base_dir, subfolders):
    """Collect *.zip files from each named sub-folder of base_dir (non-recursive).
    Prints a per-folder count so a typo'd/missing folder is obvious immediately."""
    all_paths = []
    for folder in subfolders:
        folder_path = os.path.join(base_dir, folder)
        found = sorted(glob.glob(os.path.join(folder_path, "*.zip")))
        print(f"  {folder}: {len(found)} zip(s)" + ("" if found else "  <-- none found, check the path/name"))
        all_paths.extend(found)
    return all_paths

# ---- knobs ----------------------------------------------------------------
PARAM_COLS = ["orb_minutes", "use_volume_filter", "volume_mult", "use_vwap_filter",
              "vwap_slope_lookback", "candle_strength_pct", "use_candle_filter",
              "stop_atr_mult", "target_mult", "vwap_min_slope", "use_direction_bias"]

FDR_ALPHA = 0.05          # significance threshold
SIGNIFICANCE_COL = "p-value"  # "p-value" (raw) or "p-value (FDR-corrected)" -- which column the
                               # Top Winners/Losers (Significant) tabs and Shortlist gate on.
                               # Currently RAW p-value: at large combo counts, FDR correction
                               # collapses to one conservative plateau value (see Guide tab) that lets
                               # almost nothing through. Switch this back to the FDR-corrected column
                               # once that family-scoping question is resolved.
MIN_YEARS_FOR_CONSISTENCY = 2  # a combo needs data in at least this many years before "consistency
                                # across years" is even measurable -- with only 1 year, Std is
                                # mathematically undefined (not 0), so a 1-year combo would otherwise
                                # look artificially "perfectly consistent" and get an unfair edge over
                                # combos that have actually been tested across multiple years. Combos
                                # below this are excluded from Top Winners/Losers (they still appear
                                # in the plain Ranking tab).
ROBUSTNESS_MIN = 0.55     # min fraction of profitable days for the Shortlist
PROFIT_FACTOR_MIN = 1.2   # min gross-profit/gross-loss ratio for the Shortlist
EXPECTANCY_MIN = 5.0      # min $/trade for the Shortlist
TOP_N = 30                # size of the Top Winners / Top Losers tabs
IQR_K = 1.5               # low-outlier fence multiplier on "periods tested"

# equal-weight Balance ("Composite") Score components: metric -> "higher"/"lower" is better
BALANCE_COMPONENTS = {
    "Robustness (% Profitable Days)": "higher",
    "Total (Pnl)": "higher",
    "Profit Factor": "higher",
    "Expectancy ($/trade)": "higher",
    "Average (Pnl)": "higher",
    "Error (stdev/avg %)": "lower",
}

PATH_RE = re.compile(r"sweep_results/([^/]+)/([^/]+)\.csv$")
FOOTER_RESULT_COLS = {"stopped_out", "hit_target", "closed_eod", "win_rate_%", "total_net_pnl_$"}
# trades header: symbol,bias,outcome,entry_time,entry,stop,target,stop_mult,target_mult,exit_time,exit,shares,pnl,r_multiple
SYMBOL_IDX, ENTRY_IDX, SHARES_IDX, PNL_IDX = 0, 4, 11, 12

print("Config loaded.")


# %% [CELL 2] ================================================================
# PARSING — one pass per zip (multi-process), pure string splitting (no
# per-file pandas.read_csv). Memory-safe by design for very large runs:
#   - each worker parses ONE zip, converts straight to a compact DataFrame,
#     and writes it to its own small Parquet "part" file on disk -- row data
#     is NEVER accumulated as Python dicts across zips in the parent process
#     (that accumulation is what exhausted RAM before).
#   - already-parsed zips are SKIPPED on a re-run (their part file already
#     exists), so if Colab crashes or you stop the runtime, re-running this
#     cell resumes instead of re-parsing everything from scratch.
#   - combo_days  : one row per (combo, date)   -> everything combo-level
#   - symbol_days : one row per (symbol, date), SUMMED ACROSS EVERY COMBO
#                   tested that day -- this is the granularity Symbol Stats
#                   actually needs ("conflating every combo and day
#                   together"), and collapsing the combo dimension here
#                   (rather than after the fact) is what keeps this table
#                   from being ~100-200x bigger than it needs to be.
# =============================================================================
def _cast(val):
    val = val.strip()
    if val in ("True", "False"):
        return val == "True"
    try:
        f = float(val)
        return int(f) if f.is_integer() else f
    except ValueError:
        return val


def _parse_one_zip(zip_path):
    """Returns (combo_day_rows, symbol_day_rows) for one zip.
    symbol_day_rows is keyed by (date, symbol) ONLY -- summed across every
    combo folder in this zip that traded that symbol that day."""
    combo_day_rows = []
    symbol_day_agg = {}  # (date, symbol) -> [n, wins, gross_profit, gross_loss, invested]

    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            m = PATH_RE.search(name)
            if not m:
                continue
            combo_folder, date_str = m.group(1), m.group(2)
            text = zf.read(name).decode("utf-8", errors="replace")
            lines = text.split("\n")
            blank_idx = next((i for i, l in enumerate(lines) if l.strip() == ""), None)
            trades_lines = lines[:blank_idx] if blank_idx is not None else lines
            footer_lines = [l for l in (lines[blank_idx + 1:] if blank_idx is not None else []) if l.strip()]

            n = num_wins = 0
            gross_profit = gross_loss = invested = 0.0

            for line in trades_lines[1:]:
                if not line.strip():
                    continue
                parts = line.split(",")
                if len(parts) <= PNL_IDX:
                    continue
                try:
                    entry = float(parts[ENTRY_IDX])
                    shares = float(parts[SHARES_IDX])
                    pnl = float(parts[PNL_IDX])
                except ValueError:
                    continue
                symbol = parts[SYMBOL_IDX].strip()
                is_win = pnl > 0
                inv = entry * shares

                n += 1
                invested += inv
                if is_win:
                    gross_profit += pnl
                    num_wins += 1
                elif pnl < 0:
                    gross_loss += -pnl

                s = symbol_day_agg.setdefault((date_str, symbol), [0, 0, 0.0, 0.0, 0.0])
                s[0] += 1
                s[4] += inv
                if is_win:
                    s[2] += pnl
                    s[1] += 1
                elif pnl < 0:
                    s[3] += -pnl

            footer_dict = {}
            if len(footer_lines) >= 2:
                headers = footer_lines[0].split(",")
                values = footer_lines[1].split(",")
                footer_dict = {h.strip(): _cast(v) for h, v in zip(headers, values)}

            net_pnl = gross_profit - gross_loss
            row = {"combo_folder": combo_folder, "date": date_str,
                   "num_trades": n, "num_wins": num_wins,
                   "gross_profit": gross_profit, "gross_loss": gross_loss,
                   "net_pnl_computed": net_pnl, "total_invested": invested}
            for k, v in footer_dict.items():
                row[f"footer_{k}" if k in FOOTER_RESULT_COLS else k] = v
            combo_day_rows.append(row)

    symbol_day_rows = [
        {"symbol": sym, "date": date_str, "num_trades": n, "num_wins": w,
         "gross_profit": gp, "gross_loss": gl, "net_pnl_computed": gp - gl, "total_invested": inv}
        for (date_str, sym), (n, w, gp, gl, inv) in symbol_day_agg.items()
    ]
    return combo_day_rows, symbol_day_rows


def _finalize_days(df):
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="coerce")
    df["year"] = df["date"].dt.year
    df["quarter"] = df["date"].dt.to_period("Q").astype(str)
    if "combo_folder" in df.columns:
        df["combo_folder"] = df["combo_folder"].astype("category")
    if "symbol" in df.columns:
        df["symbol"] = df["symbol"].astype("category")
    for c in df.select_dtypes(include="float64").columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    for c in df.select_dtypes(include="int64").columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df


# ---- per-zip Parquet part cache (this is what makes parsing resumable and
#      keeps peak memory bounded to ONE zip's worth of data at a time) ------
PARTS_DIR = os.path.join(MASTER_DIR, "parts")
COMBO_PARTS_DIR = os.path.join(PARTS_DIR, "combo_days")
SYMBOL_PARTS_DIR = os.path.join(PARTS_DIR, "symbol_days")
os.makedirs(COMBO_PARTS_DIR, exist_ok=True)
os.makedirs(SYMBOL_PARTS_DIR, exist_ok=True)


def _part_path(zip_path, parts_dir):
    # folder name is included so two zips with the same filename in
    # different SOURCE_FOLDERS don't collide
    safe = zip_path.replace(BASE_DIR, "").strip("/").replace("/", "__")
    return os.path.join(parts_dir, os.path.splitext(safe)[0] + ".parquet")


def _parse_and_cache_one_zip(zip_path):
    """Runs in a worker process. Parses one zip and writes its two compact
    Parquet parts to disk; returns only small metadata (never the row data
    itself) back to the parent, so IPC payload stays tiny regardless of
    how many rows the zip contained."""
    combo_part = _part_path(zip_path, COMBO_PARTS_DIR)
    symbol_part = _part_path(zip_path, SYMBOL_PARTS_DIR)

    if os.path.exists(combo_part) and os.path.exists(symbol_part):
        import pyarrow.parquet as pq
        n_combo = pq.ParquetFile(combo_part).metadata.num_rows
        n_symbol = pq.ParquetFile(symbol_part).metadata.num_rows
        return zip_path, n_combo, n_symbol, True

    combo_rows, symbol_rows = _parse_one_zip(zip_path)
    combo_df = _finalize_days(pd.DataFrame(combo_rows)) if combo_rows else pd.DataFrame()
    symbol_df = _finalize_days(pd.DataFrame(symbol_rows)) if symbol_rows else pd.DataFrame()
    combo_df.to_parquet(combo_part, compression="snappy")
    symbol_df.to_parquet(symbol_part, compression="snappy")
    return zip_path, len(combo_df), len(symbol_df), False


def parse_zips(zip_paths, max_workers=None):
    if not zip_paths:
        raise FileNotFoundError(
            f"No .zip files found under {BASE_DIR} in folders {SOURCE_FOLDERS}. "
            "Check BASE_DIR/SOURCE_FOLDERS in Cell 1 and that Drive is mounted.")
    t0 = time.time()
    max_workers = max_workers or min(8, os.cpu_count() or 4)
    print(f"Parsing {len(zip_paths)} zip(s) with {max_workers} worker process(es)...")
    n_cached = n_parsed = total_combo_rows = total_symbol_rows = 0
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_parse_and_cache_one_zip, zp): zp for zp in zip_paths}
        for fut in as_completed(futures):
            zp = futures[fut]
            zip_path, n_combo, n_symbol, was_cached = fut.result()
            n_cached += was_cached
            n_parsed += (not was_cached)
            total_combo_rows += n_combo
            total_symbol_rows += n_symbol
            tag = "cached" if was_cached else "parsed"
            print(f"  [{tag}] {os.path.basename(zip_path)}: {n_combo} combo-day rows, "
                  f"{n_symbol} symbol-day rows")
    print(f"Done in {time.time() - t0:.1f}s ({n_parsed} newly parsed, {n_cached} already cached). "
          f"Total: {total_combo_rows:,} combo-day rows, {total_symbol_rows:,} symbol-day rows.")


# ---- run parsing (skips zips whose part file already exists) --------------
print(f"Scanning {len(SOURCE_FOLDERS)} folder(s) under {BASE_DIR}:")
zip_paths = gather_zip_paths(BASE_DIR, SOURCE_FOLDERS)
parse_zips(zip_paths)

# ---- combine all part files into the two master tables --------------------
# This is the only point the full dataset is held in memory at once, and by
# now it's a compact columnar DataFrame (not a list of dicts), so it's a
# fraction of the size that caused the earlier crash.
combo_parts = sorted(glob.glob(os.path.join(COMBO_PARTS_DIR, "*.parquet")))
symbol_parts = sorted(glob.glob(os.path.join(SYMBOL_PARTS_DIR, "*.parquet")))
combo_days = pd.concat([pd.read_parquet(p) for p in combo_parts], ignore_index=True)

# symbol_days: collapse any (symbol, date) that appears in more than one zip
# (e.g. overlapping "_V2" re-run folders) so that calendar day isn't counted
# twice in Symbol Stats' day-count / robustness / p-value.
symbol_days_raw = pd.concat([pd.read_parquet(p) for p in symbol_parts], ignore_index=True)
symbol_days = symbol_days_raw.groupby(["symbol", "date"], observed=True, as_index=False).agg(
    num_trades=("num_trades", "sum"), num_wins=("num_wins", "sum"),
    gross_profit=("gross_profit", "sum"), gross_loss=("gross_loss", "sum"),
    net_pnl_computed=("net_pnl_computed", "sum"), total_invested=("total_invested", "sum"))
symbol_days = _finalize_days(symbol_days)
del symbol_days_raw

combo_days.to_parquet(os.path.join(MASTER_DIR, "combo_days.parquet"), compression="snappy")
symbol_days.to_parquet(os.path.join(MASTER_DIR, "symbol_days.parquet"), compression="snappy")

print(f"combo_days: {len(combo_days):,} rows, {combo_days['combo_folder'].nunique():,} combos, "
      f"{combo_days['date'].nunique():,} days, years {sorted(combo_days['year'].dropna().unique())}")
print(f"symbol_days: {len(symbol_days):,} rows, {symbol_days['symbol'].nunique():,} symbols")


# %% [CELL 3] ================================================================
# COMBO IDS — short, stable labels (Combo001, Combo002...) so every tab
# cross-references easily instead of long folder-name strings.
#
# IMPORTANT: Combo ID identity is now based on the PARSED, TYPED parameter
# values (orb_minutes=15, target_mult=3.0, ...) rather than the raw
# combo_folder string. This matters because if the same parameter combo was
# named with any formatting drift between runs/years (e.g. "target_mult=3"
# one year vs "target_mult=3.0" another), matching on the raw string would
# silently treat them as two unrelated combos -- which breaks cross-year
# consistency (every combo would look like it only ever ran once) and
# quietly loses data. Matching on parsed VALUES fixes that.
#
# This also populates the individual parameter columns (orb_minutes,
# use_volume_filter, ...) that Parameter Performance / Sensitivity need --
# previously these only existed baked into the combo_folder string and were
# never actually split out, so those tabs would have been empty.
#
# The parser below is ORDER- and SUBSET-agnostic: it looks for each known
# PARAM_COLS name wherever it appears in the string, in whatever order, and
# simply doesn't set a value for a param that isn't present in a given
# folder name. This matters because different experiment families
# (RUNS_Orb_15_30_..., RUNS_Orb_10_20_..., RUNS_Orb5_...) may not all test
# the exact same parameter set or write them in the exact same order -- an
# exact full-string match would silently fail (and drop parsing entirely)
# for any family that differs even slightly from the first one.
# =============================================================================
# longest names first, so e.g. "vwap_slope_lookback" isn't mistaken for a
# partial match against a shorter overlapping name
_PARAM_NAMES_BY_LEN = sorted(PARAM_COLS, key=len, reverse=True)
_KV_PATTERN = re.compile(r"(?:^|_)(" + "|".join(re.escape(p) for p in _PARAM_NAMES_BY_LEN) + r")=")


def _typed_value(raw):
    raw = raw.strip().strip("_")
    if raw.upper() in ("T", "TRUE"):
        return True
    if raw.upper() in ("F", "FALSE"):
        return False
    try:
        return float(raw)
    except ValueError:
        return raw


def parse_combo_folder(folder):
    """Extract whatever known PARAM_COLS key=value pairs appear in this
    string, in any order, with any subset present. Returns None only if
    NONE of the known parameter names were found at all."""
    folder = str(folder)
    matches = list(_KV_PATTERN.finditer(folder))
    if not matches:
        return None
    parsed = {}
    for i, m in enumerate(matches):
        key = m.group(1)
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(folder)
        parsed[key] = _typed_value(folder[start:end])
    return parsed


def combo_key(parsed):
    """Canonical identity string built from typed VALUES, not raw text --
    e.g. target_mult=3 and target_mult=3.0 both become 'target_mult=3'.
    A param absent from this combo's string is marked NA (not silently
    skipped), so combos that genuinely test different parameter sets are
    never accidentally treated as identical."""
    parts = []
    for p in PARAM_COLS:
        v = parsed.get(p, "NA")
        if isinstance(v, bool):
            parts.append(f"{p}={'T' if v else 'F'}")
        elif isinstance(v, float):
            parts.append(f"{p}={v:.6g}")
        else:
            parts.append(f"{p}={v}")
    return "_".join(parts)


combo_days["combo_folder"] = combo_days["combo_folder"].astype(str)
unique_folders = combo_days["combo_folder"].unique()
parsed_lookup = {f: parse_combo_folder(f) for f in unique_folders}
unmatched = [f for f, p in parsed_lookup.items() if p is None]
if unmatched:
    print(f"WARNING: {len(unmatched)} combo_folder value(s) contained NONE of the known "
          f"PARAM_COLS names at all (check PARAM_COLS in Cell 1 covers every parameter "
          f"your folder names use) -- these will each be treated as their own combo, "
          f"with no parsed parameter columns. Examples: {unmatched[:3]}")

partial = [f for f, p in parsed_lookup.items()
           if p is not None and set(p.keys()) != set(PARAM_COLS)]
if partial:
    missing_examples = {f: sorted(set(PARAM_COLS) - set(parsed_lookup[f].keys())) for f in partial[:3]}
    print(f"NOTE: {len(partial)} combo_folder value(s) were missing one or more PARAM_COLS "
          f"(likely a different experiment family testing a different parameter set) -- "
          f"those params are recorded as NA for those combos, not silently dropped. "
          f"Examples of what's missing: {missing_examples}")

combo_days["combo_key"] = combo_days["combo_folder"].map(
    lambda f: combo_key(parsed_lookup[f]) if parsed_lookup[f] is not None else f)

# attach individual parsed parameter columns (needed for Parameter Performance/Sensitivity)
matched_folders = [f for f in unique_folders if parsed_lookup[f] is not None]
if matched_folders:
    param_df = pd.DataFrame({f: parsed_lookup[f] for f in matched_folders}).T
    param_df.index.name = "combo_folder"
    combo_days = combo_days.merge(param_df.reset_index(), on="combo_folder", how="left")
    print(f"Parsed parameter columns attached to combo_days: {list(param_df.columns)}")
else:
    print("Parsed parameter columns attached to combo_days: NONE "
          "(every combo_folder value matched none of the known PARAM_COLS names -- "
          "see the WARNING above).")


def assign_combo_ids(df):
    keys = sorted(df["combo_key"].unique())
    id_map = {k: f"Combo{i+1:03d}" for i, k in enumerate(keys)}  # Combo001, Combo002, ...
    # display string: the first-seen raw combo_folder for each canonical key
    first_seen = df.drop_duplicates(subset=["combo_key"]).set_index("combo_key")["combo_folder"]
    combo_map = pd.DataFrame({"Combo ID": [id_map[k] for k in keys],
                               "Combo Parameters": [first_seen.loc[k] for k in keys]})
    return id_map, combo_map


id_map, combo_map = assign_combo_ids(combo_days)
combo_days["Combo ID"] = combo_days["combo_key"].map(id_map)
print(f"{len(combo_map)} combos assigned IDs Combo001 .. {combo_map['Combo ID'].iloc[-1]}")

n_raw_folders = combo_days["combo_folder"].nunique()
n_canonical = combo_days["combo_key"].nunique()
if n_raw_folders != n_canonical:
    print(f"NOTE: collapsed {n_raw_folders} raw combo_folder strings into {n_canonical} "
          f"canonical combos (some were the same parameters with different text formatting).")
    if n_canonical < n_raw_folders * 0.5:
        print(f"CAUTION: that's a large collapse ({n_raw_folders} -> {n_canonical}, more than 2x). "
              f"Worth a sanity check that PARAM_COLS (Cell 1) actually lists EVERY parameter your "
              f"folder names vary -- if one is missing from that list, genuinely different combos "
              f"that only differ in that missing parameter would get merged here by mistake.")


# %% [CELL 4] ================================================================
# METRICS ENGINE — same math as your original compute_full_metrics, usable
# for any grouping: [Combo ID], [Combo ID, year], [Combo ID, quarter] ...
# =============================================================================
def compute_full_metrics(df, group_cols, param_cols=PARAM_COLS):
    param_cols = [c for c in param_cols if c in df.columns]
    out_rows = []
    for keys, g in df.groupby(group_cols, observed=True):
        daily_pnl = g["net_pnl_computed"].values
        total_trades = g["num_trades"].sum()
        total_wins = g["num_wins"].sum()
        gross_profit, gross_loss = g["gross_profit"].sum(), g["gross_loss"].sum()
        total_invested = g["total_invested"].sum()
        hit_target = g["footer_hit_target"].sum() if "footer_hit_target" in g.columns else np.nan

        avg_pnl = daily_pnl.mean()
        std_pnl = daily_pnl.std(ddof=1) if len(daily_pnl) > 1 else np.nan
        median_pnl = np.median(daily_pnl)
        total_pnl = daily_pnl.sum()

        if len(daily_pnl) > 1 and std_pnl and std_pnl > 0:
            _, p_val = stats.ttest_1samp(daily_pnl, popmean=0)
        else:
            p_val = np.nan

        key_tuple = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_cols, key_tuple))
        row.update({
            "Total (Pnl)": total_pnl, "Average (Pnl)": avg_pnl, "Standard Deviation": std_pnl,
            "Error (stdev/avg %)": (std_pnl / avg_pnl * 100) if avg_pnl else np.nan,
            "Median (Pnl)": median_pnl, "Total Trades": int(total_trades),
            "Hit Target %": (hit_target / total_trades * 100) if total_trades else np.nan,
            "Win rate %": (total_wins / total_trades * 100) if total_trades else np.nan,
            "Total Invested ($)": total_invested,
            "Profit Factor": (gross_profit / gross_loss) if gross_loss else np.inf,
            "Risk-Adjusted (Sharpe-like)": (avg_pnl / std_pnl) if std_pnl else np.nan,
            "Robustness (% Profitable Days)": (daily_pnl > 0).mean(),
            "p-value": p_val, "Days Tested": len(daily_pnl),
            "First Seen": g["date"].min(), "Last Seen": g["date"].max(),
        })
        if "Combo ID" in df.columns:
            for pc in param_cols:
                row[pc] = g[pc].iloc[0]
        out_rows.append(row)

    out = pd.DataFrame(out_rows)
    valid = out["p-value"].notna()
    corrected = pd.Series(np.nan, index=out.index)
    if valid.sum():
        _, pv, _, _ = multipletests(out.loc[valid, "p-value"], method="fdr_bh")
        corrected.loc[valid] = pv
    out["p-value (FDR-corrected)"] = corrected
    out["Return on Capital %"] = out["Total (Pnl)"] / out["Total Invested ($)"] * 100
    out["Expectancy ($/trade)"] = out["Total (Pnl)"] / out["Total Trades"]
    return out.sort_values("Total (Pnl)", ascending=False).reset_index(drop=True)


# Total (all years combined), Yearly, and Quarterly breakdowns — req. #6
total_metrics = compute_full_metrics(combo_days, ["Combo ID"])
yearly_metrics = compute_full_metrics(combo_days, ["Combo ID", "year"])
quarterly_metrics = compute_full_metrics(combo_days, ["Combo ID", "quarter"])
print(f"Total: {len(total_metrics)} combos | Yearly rows: {len(yearly_metrics)} | "
      f"Quarterly rows: {len(quarterly_metrics)}")


# %% [CELL 5] ================================================================
# BALANCE ("COMPOSITE") SCORE — per your definition: equal-weight blend of
# Robustness, Total PnL, Profit Factor, Expectancy, Average PnL, and
# (inverted) Error%. Applied at all three grains you asked for:
#   Total    -> one score per combo (ranked against every other combo, once)
#   Yearly   -> scored per combo per year, then rolled into Mean/Std/Min/Max
#               + Consistent Winner/Loser Score across years
#   Quarterly-> same roll-up, across quarters instead of years
# =============================================================================
def add_balance_score(period_metrics, period_col):
    df = period_metrics.copy()
    ranks = pd.DataFrame(index=df.index)
    for col, direction in BALANCE_COMPONENTS.items():
        vals = df[col].replace([np.inf, -np.inf], np.nan)
        period_best = vals.groupby(df[period_col]).transform("max" if direction == "higher" else "min")
        filled = df[col].where(np.isfinite(df[col]), period_best)
        if direction == "lower":
            filled = -filled
        ranks[col] = filled.groupby(df[period_col]).rank(pct=True, na_option="keep")
    df["Balance Score"] = ranks.mean(axis=1, skipna=True)
    return df


def consistency_from_periods(period_scored, suffix, id_col="Combo ID", score_col="Balance Score"):
    g = period_scored.groupby(id_col, observed=True)[score_col]
    out = g.agg(Mean="mean", Std="std", Min="min", Max="max", Periods_Tested="count").reset_index()
    out["Std"] = out["Std"].fillna(0.0)
    out[f"Consistent Winner Score ({suffix})"] = out["Mean"] - out["Std"]   # reliably good
    out[f"Consistent Loser Score ({suffix})"] = out["Mean"] + out["Std"]    # reliably bad
    return out.rename(columns={
        "Mean": f"Mean Composite Score ({suffix})", "Std": f"Std Composite Score ({suffix})",
        "Min": f"Min Composite Score ({suffix})", "Max": f"Max Composite Score ({suffix})",
        "Periods_Tested": f"Periods Tested ({suffix})"})


def exclude_low_session_outliers(df, count_col, k=IQR_K):
    """Drop combos with an abnormally LOW number of periods tested
    (IQR x1.5 low-fence rule) so thin-data combos can't fake consistency."""
    q1, q3 = df[count_col].quantile([0.25, 0.75])
    low_fence = q1 - k * (q3 - q1)
    kept = df[df[count_col] >= low_fence].copy()
    return kept, len(df) - len(kept), low_fence


# Total: one Balance Score per combo, ranked against every other combo once
total_scored = add_balance_score(total_metrics.assign(_all="ALL"), "_all").drop(columns="_all")
total_scored = total_scored.rename(columns={"Balance Score": "Balance Score (Total)"})

# Yearly and Quarterly: scored within each period, then rolled up into consistency stats
yearly_scored = add_balance_score(yearly_metrics, "year")
quarterly_scored = add_balance_score(quarterly_metrics, "quarter")
consistency_yearly = consistency_from_periods(yearly_scored, "Yearly")
consistency_quarterly = consistency_from_periods(quarterly_scored, "Quarterly")

combo_metrics = (total_scored
                  .merge(consistency_yearly, on="Combo ID", how="left")
                  .merge(consistency_quarterly, on="Combo ID", how="left")
                  .merge(combo_map, on="Combo ID", how="left"))

# For the Top-30 consistency tabs: first, a HARD floor -- a combo needs at
# least MIN_YEARS_FOR_CONSISTENCY years of data before "consistency across
# years" is measurable at all (with 1 year, Std is undefined, not 0 --
# defaulting it to 0 would make under-tested combos look artificially
# perfect). Then, among what's left, drop any remaining low-session
# outliers via IQR (catches e.g. a combo with 2 years when everyone else
# has 4-5).
n_before_floor = len(combo_metrics)
combo_metrics_kept = combo_metrics[combo_metrics["Periods Tested (Yearly)"] >= MIN_YEARS_FOR_CONSISTENCY].copy()
n_dropped_floor = n_before_floor - len(combo_metrics_kept)
print(f"Excluded {n_dropped_floor} combo(s) with fewer than {MIN_YEARS_FOR_CONSISTENCY} years tested "
      f"(consistency across years isn't measurable from a single year).")

combo_metrics_kept, n_dropped_iqr, low_fence = exclude_low_session_outliers(combo_metrics_kept, "Periods Tested (Yearly)")
if n_dropped_iqr:
    print(f"Additionally excluded {n_dropped_iqr} combo(s) as low-session outliers "
          f"(fewer than {low_fence:.1f} years tested, relative to peers).")


# %% [CELL 6] ================================================================
# RANKING TAB — one row per combo, Combo ID first, matches the column layout
# you specified.
# =============================================================================
RANKING_COLS = [
    "Combo ID", "Combo Parameters", "Balance Score (Total)",
    "Mean Composite Score (Yearly)", "Std Composite Score (Yearly)",
    "Min Composite Score (Yearly)", "Max Composite Score (Yearly)",
    "Consistent Winner Score (Yearly)", "Consistent Loser Score (Yearly)",
    "Periods Tested (Yearly)",
    "Mean Composite Score (Quarterly)", "Std Composite Score (Quarterly)",
    "Min Composite Score (Quarterly)", "Max Composite Score (Quarterly)",
    "Consistent Winner Score (Quarterly)", "Consistent Loser Score (Quarterly)",
    "Periods Tested (Quarterly)",
    "Total (Pnl)", "Average (Pnl)", "Standard Deviation", "Error (stdev/avg %)",
    "Median (Pnl)", "Total Trades", "Hit Target %", "Win rate %", "Total Invested ($)",
    "Profit Factor", "Risk-Adjusted (Sharpe-like)", "Robustness (% Profitable Days)",
    "p-value", "p-value (FDR-corrected)", "Return on Capital %", "Expectancy ($/trade)",
]
ranking_tab = combo_metrics[[c for c in RANKING_COLS if c in combo_metrics.columns]] \
    .sort_values("Consistent Winner Score (Yearly)", ascending=False).reset_index(drop=True)
print(ranking_tab.head(10).to_string(index=False))


# %% [CELL 7] ================================================================
# TOP 30 WINNERS / LOSERS BY CONSISTENCY (low-session outliers excluded),
# plus the "Significant" variants filtered to FDR-corrected p <= 0.05.
# =============================================================================
top_winners = combo_metrics_kept.sort_values("Consistent Winner Score (Yearly)", ascending=False).head(TOP_N)
top_losers = combo_metrics_kept.sort_values("Consistent Loser Score (Yearly)", ascending=True).head(TOP_N)

sig = combo_metrics_kept[combo_metrics_kept[SIGNIFICANCE_COL] <= FDR_ALPHA]
top_winners_sig = sig.sort_values("Consistent Winner Score (Yearly)", ascending=False).head(TOP_N)
top_losers_sig = sig.sort_values("Consistent Loser Score (Yearly)", ascending=True).head(TOP_N)

DETAIL_COLS = ["Combo ID", "Combo Parameters", "Periods Tested (Yearly)", "Mean Composite Score (Yearly)",
               "Std Composite Score (Yearly)", "Consistent Winner Score (Yearly)",
               "Consistent Loser Score (Yearly)",
               "Total (Pnl)", "Robustness (% Profitable Days)", "p-value", "p-value (FDR-corrected)",
               "Return on Capital %"]
top_winners = top_winners[[c for c in DETAIL_COLS if c in top_winners.columns]]
top_losers = top_losers[[c for c in DETAIL_COLS if c in top_losers.columns]]
top_winners_sig = top_winners_sig[[c for c in DETAIL_COLS if c in top_winners_sig.columns]]
top_losers_sig = top_losers_sig[[c for c in DETAIL_COLS if c in top_losers_sig.columns]]
print(f"Winners: {len(top_winners)}, Losers: {len(top_losers)}, "
      f"Significant Winners: {len(top_winners_sig)}, Significant Losers: {len(top_losers_sig)}")


# %% [CELL 8] ================================================================
# PARAMETER PERFORMANCE / BEST / WORST / SENSITIVITY — how much does each
# knob move the needle, holding everything else as tested.
# =============================================================================
def parameter_performance(combo_metrics_total, param_cols=PARAM_COLS,
                           value_cols=("Balance Score (Total)", "Total (Pnl)", "Win rate %",
                                       "Robustness (% Profitable Days)", "Return on Capital %")):
    param_cols = [c for c in param_cols if c in combo_metrics_total.columns]
    value_cols = [c for c in value_cols if c in combo_metrics_total.columns]
    long_rows = []
    for pc in param_cols:
        g = combo_metrics_total.groupby(pc, observed=True)[value_cols].mean().reset_index()
        g.insert(0, "Parameter", pc)
        g = g.rename(columns={pc: "Value"})
        g["Combos"] = combo_metrics_total.groupby(pc, observed=True).size().values
        long_rows.append(g)
    perf = pd.concat(long_rows, ignore_index=True) if long_rows else pd.DataFrame()

    sens_rows = []
    for pc in param_cols:
        means = combo_metrics_total.groupby(pc, observed=True)["Balance Score (Total)"].mean()
        if len(means) > 1:
            sens_rows.append({"Parameter": pc,
                               "Sensitivity (Balance Score range)": means.max() - means.min(),
                               "Best Value": means.idxmax(), "Worst Value": means.idxmin()})
    sensitivity = (pd.DataFrame(sens_rows)
                   .sort_values("Sensitivity (Balance Score range)", ascending=False)
                   .reset_index(drop=True)) if sens_rows else pd.DataFrame()

    best = (perf.sort_values("Balance Score (Total)", ascending=False)
            .groupby("Parameter", observed=True).head(1).reset_index(drop=True)) if len(perf) else perf
    worst = (perf.sort_values("Balance Score (Total)", ascending=True)
             .groupby("Parameter", observed=True).head(1).reset_index(drop=True)) if len(perf) else perf
    return perf, best, worst, sensitivity


param_perf, param_best, param_worst, param_sensitivity = parameter_performance(combo_metrics)
_found_param_cols = [c for c in PARAM_COLS if c in combo_metrics.columns]
print(f"Parameter columns found on combo_metrics: {_found_param_cols if _found_param_cols else 'NONE'}")
if not _found_param_cols:
    print("Parameter Performance/Best/Worst/Sensitivity will be EMPTY because none of PARAM_COLS "
          "made it onto combo_metrics. Scroll up to Cell 3's output -- if you see a WARNING there "
          "about combo_folder values matching NONE of the known PARAM_COLS names, that's the cause: "
          "either PARAM_COLS (Cell 1) doesn't match your actual folder-naming scheme at all, or "
          "combo_days ended up without a usable 'combo_folder' column. If Cell 3 printed no warning "
          "and combos WERE assigned IDs normally, but this still says NONE, that points to something "
          "further upstream -- share the full Cell 3 output and this line for a definitive diagnosis.")
print(param_sensitivity.to_string(index=False) if len(param_sensitivity) else "No PARAM_COLS found in data.")


# %% [CELL 9] ================================================================
# SHORTLIST — combos that pass ALL of: FDR-significant, robust (>=55% days
# profitable), a real edge (Profit Factor >= 1.2), net positive per trade
# (Expectancy >= $0), ranked by Return on Capital %. "Which combo can I
# trust." NOTE: the FDR-significance filter is known to be very strict at
# this combo count right now (see the Guide tab) -- left in place as
# requested, pending a fix to how the correction family is scoped.
# =============================================================================
shortlist = combo_metrics[
    (combo_metrics[SIGNIFICANCE_COL] <= FDR_ALPHA) &
    (combo_metrics["Robustness (% Profitable Days)"] >= ROBUSTNESS_MIN) &
    (combo_metrics["Profit Factor"] >= PROFIT_FACTOR_MIN) &
    (combo_metrics["Expectancy ($/trade)"] >= EXPECTANCY_MIN)
].sort_values("Return on Capital %", ascending=False).reset_index(drop=True)
shortlist = shortlist[[c for c in RANKING_COLS if c in shortlist.columns]]
print(f"Shortlist: {len(shortlist)} combo(s) pass all filters.")


# %% [CELL 10] ===============================================================
# SYMBOL STATS — one row per symbol (conflating every combo/day). Answers
# "is this symbol worth keeping in BT_SYMBOLS at all".
# =============================================================================
symbol_stats = compute_full_metrics(
    symbol_days.rename(columns={"symbol": "Symbol"}), ["Symbol"])

SYMBOL_COLS = ["Symbol", "Total (Pnl)", "Average (Pnl)", "Win rate %", "Profit Factor",
               "Robustness (% Profitable Days)", "Total Trades", "Return on Capital %",
               "p-value", "p-value (FDR-corrected)"]
symbol_stats = symbol_stats[[c for c in SYMBOL_COLS if c in symbol_stats.columns]] \
    .sort_values("Total (Pnl)", ascending=False).reset_index(drop=True)
print(f"Symbol Stats: {len(symbol_stats)} symbols.")


# %% [CELL 11] ===============================================================
# TOTAL / YEARLY / QUARTERLY TABS — long format, one row per (combo, period),
# each carrying the full metric set (not a trimmed subset).
# =============================================================================
FULL_METRIC_COLS = [
    "Days Tested", "Total (Pnl)", "Average (Pnl)", "Standard Deviation", "Error (stdev/avg %)",
    "Median (Pnl)", "Total Trades", "Hit Target %", "Win rate %", "Total Invested ($)",
    "Profit Factor", "Risk-Adjusted (Sharpe-like)", "Robustness (% Profitable Days)",
    "p-value", "p-value (FDR-corrected)", "Return on Capital %", "Expectancy ($/trade)",
]

yearly_tab = yearly_scored.merge(combo_map, on="Combo ID")[
    ["Combo ID", "Combo Parameters", "year"] +
    [c for c in FULL_METRIC_COLS if c in yearly_scored.columns] + ["Balance Score"]
].sort_values(["year", "Total (Pnl)"], ascending=[True, False]).reset_index(drop=True)

quarterly_tab = quarterly_scored.merge(combo_map, on="Combo ID")[
    ["Combo ID", "Combo Parameters", "quarter"] +
    [c for c in FULL_METRIC_COLS if c in quarterly_scored.columns] + ["Balance Score"]
].sort_values(["quarter", "Total (Pnl)"], ascending=[True, False]).reset_index(drop=True)

total_tab = total_scored.merge(combo_map, on="Combo ID")[
    ["Combo ID", "Combo Parameters"] +
    [c for c in FULL_METRIC_COLS if c in total_scored.columns] + ["Balance Score (Total)"]
].sort_values("Total (Pnl)", ascending=False).reset_index(drop=True)


# %% [CELL 12] ===============================================================
# GUIDE TAB — plain-English glossary of every tab and metric, so this is
# self-explanatory to anyone opening the workbook cold.
# =============================================================================
guide_rows = [
    ("Tabs", "Guide", "This sheet. What every tab and column means."),
    ("Tabs", "Ranking", "One row per combo (Combo ID + Combo Parameters), full metric set at "
                         "Total/Yearly/Quarterly granularity, sorted by Consistent Winner Score (Yearly). "
                         "Your main leaderboard."),
    ("Tabs", "Top Winners", f"Top {TOP_N} combos by Consistent Winner Score (Yearly). Combos with fewer "
                             f"than {MIN_YEARS_FOR_CONSISTENCY} years of data are excluded entirely -- "
                             "consistency isn't measurable from a single year -- and any remaining "
                             "low-session outliers (IQR x1.5 rule) are excluded too."),
    ("Tabs", "Top Losers", f"Mirror of Top Winners: top {TOP_N} combos by Consistent Loser Score (Yearly) "
                            "-- reliably bad, not just unlucky once."),
    ("Tabs", "Top Winners (Significant)", "Same as Top Winners, additionally filtered to combos whose "
                                           f"{SIGNIFICANCE_COL} <= {FDR_ALPHA}. With very little data "
                                           "(e.g. a single test year) this can legitimately come back empty "
                                           "-- it means nothing has cleared significance yet, not that "
                                           "anything is broken."),
    ("Tabs", "Top Losers (Significant)", "Same idea as Top Winners (Significant), for the loser side."),
    ("Tabs", "Shortlist", f"Combos that pass ALL of: significant ({SIGNIFICANCE_COL} <= {FDR_ALPHA}), "
                           f"robust (>= {int(ROBUSTNESS_MIN*100)}% of days profitable), a real edge "
                           f"(Profit Factor >= {PROFIT_FACTOR_MIN}), net positive per trade "
                           f"(Expectancy >= ${EXPECTANCY_MIN}/trade), ranked by Return on Capital %. The "
                           "answer to 'which combo should I actually trust', not just 'which combo "
                           "has the highest average'."),
    ("Tabs", "Parameter Performance", "For each parameter (e.g. orb_minutes) and each value it was tested "
                                       "with, the average Balance Score / PnL / Win rate / Robustness / ROC% "
                                       "across every combo that used that value."),
    ("Tabs", "Best / Worst Parameters", "For each parameter, the single value with the highest / lowest "
                                         "average Balance Score."),
    ("Tabs", "Parameter Sensitivity", "How much each parameter matters: the spread (max-min) of average "
                                       "Balance Score across that parameter's tested values. A high number "
                                       "means changing this knob swings performance a lot; a low number "
                                       "means it barely matters."),
    ("Tabs", "Total", "One row per combo, metrics computed over ALL data combined (every year/quarter you "
                       "fed in)."),
    ("Tabs", "Yearly", "One row per (combo, year) -- lets you see a combo's performance year by year."),
    ("Tabs", "Quarterly", "One row per (combo, quarter) -- finer-grained version of Yearly."),
    ("Tabs", "Total/Yearly/Quarterly -- what 'Average' and 'Standard Deviation' mean",
             "The underlying observation is always ONE TRADING DAY's net PnL for that combo -- that never "
             "changes. What changes between these three tabs is only which days get pooled into the "
             "average: Total pools every day across your whole multi-year history; Yearly pools only "
             "that one year's days; Quarterly pools only that one quarter's days. So 'Average (Pnl)' is "
             "always a DAILY average, never a yearly or quarterly total -- for the actual period total, "
             "look at 'Total (Pnl)' in the same row. 'Days Tested' in each row tells you how many days "
             "that average/stdev were actually built from -- an average from 3 days means something "
             "very different from one built from 60, so check it before trusting a Yearly or Quarterly "
             "row."),
    ("Tabs", "Symbol Stats", "One row per symbol, combining every combo and day together. Answers 'is this "
                              "symbol worth keeping at all', not 'how does it do under my best combo'."),
    ("Combo Identity", "Combo ID", "Short stable label (Combo001, Combo002, ...) used everywhere instead "
                                    "of the long parameter string, so tabs cross-reference easily."),
    ("Combo Identity", "Combo Parameters", "The raw combo folder name / identifier -- effectively the "
                                            "parameter combination that produced this row (e.g. which "
                                            "orb_minutes, filters, multipliers were used). Kept narrow in "
                                            "these sheets; widen the column if you need to read it in full."),
    ("Composite Score", "Balance Score", "0-1 blended score (equal weight) of Robustness, Total PnL, Profit "
                                          "Factor, Expectancy, Average PnL, and (inverted) Error %. Each "
                                          "component is converted to a percentile rank against the combo's "
                                          "PEERS IN THE SAME PERIOD before averaging, so it's not dominated "
                                          "by one spectacular metric or by a large-dollar-value outlier."),
    ("Composite Score", "Mean/Std/Min/Max Composite Score (Yearly / Quarterly)", "The Balance Score's mean, "
                                          "standard deviation, min and max across a combo's tested years or "
                                          "quarters -- i.e. how consistent that combo's blended performance "
                                          "is over time."),
    ("Composite Score", "Consistent Winner Score", "Mean - Std of the Balance Score across periods. "
                                          "Penalizes combos whose performance swings a lot; rewards combos "
                                          "that are reliably good every period."),
    ("Composite Score", "Consistent Loser Score", "Mean + Std of the Balance Score across periods. The "
                                          "mirror of Consistent Winner Score: ranks low for combos that are "
                                          "reliably bad, not just occasionally terrible."),
    ("Composite Score", "Periods Tested (Yearly / Quarterly)", "How many distinct years / quarters this "
                                          "combo has data for. Low values are the ones excluded from Top "
                                          "Winners/Losers by the IQR outlier rule."),
    ("Core Metrics", "Days Tested", "How many distinct trading days fed into this row's Average/Standard "
                                     "Deviation/Robustness/p-value. A Yearly or Quarterly row built from "
                                     "very few days should be trusted less than one built from many."),
    ("Core Metrics", "Total (Pnl)", "Sum of net PnL across every day in the period."),
    ("Core Metrics", "Average (Pnl)", "Mean daily net PnL."),
    ("Core Metrics", "Standard Deviation", "Standard deviation of daily net PnL (day-to-day volatility)."),
    ("Core Metrics", "Error (stdev/avg %)", "Standard Deviation / Average PnL, as a percent -- a rough "
                                             "'noise relative to signal' gauge. Lower is more consistent."),
    ("Core Metrics", "Median (Pnl)", "Median daily net PnL -- less sensitive to a single huge day than the "
                                      "average."),
    ("Core Metrics", "Total Trades", "Total number of individual trades across the period."),
    ("Core Metrics", "Hit Target %", "Percent of days the footer reported the target as hit."),
    ("Core Metrics", "Win rate %", "Percent of individual trades that were profitable."),
    ("Core Metrics", "Total Invested ($)", "Sum of (entry price x shares) across every trade -- capital put "
                                            "to work, not capital at risk."),
    ("Core Metrics", "Profit Factor", "Gross profit / gross loss. >1 means more won than lost; treated as "
                                       "infinite when there were zero losing trades."),
    ("Core Metrics", "Risk-Adjusted (Sharpe-like)", "Average PnL / Standard Deviation of daily PnL -- higher "
                                                      "is better return per unit of day-to-day volatility."),
    ("Core Metrics", "Robustness (% Profitable Days)", "Percent of days with positive net PnL."),
    ("Core Metrics", "p-value", "One-sample t-test that daily PnL is different from zero. Lower = more "
                                 "confidence the average PnL isn't just noise."),
    ("Core Metrics", "p-value (FDR-corrected)", "p-value adjusted (Benjamini-Hochberg) for the fact that "
                                                 "many combos were tested at once. With several thousand "
                                                 "combos, this correction is currently very conservative -- "
                                                 "many different raw p-values can end up mapped to the same "
                                                 "corrected number (a known, correct artifact of how the "
                                                 "Benjamini-Hochberg method enforces monotonicity, not a "
                                                 "bug). This is being revisited; treat this column with "
                                                 "caution for now and lean on Consistent Winner Score, "
                                                 "Profit Factor, and Robustness instead."),
    ("Core Metrics", "Return on Capital %", "Total (Pnl) / Total Invested ($) x 100."),
    ("Core Metrics", "Expectancy ($/trade)", "Total (Pnl) / Total Trades -- average profit per trade."),
    ("Config used", "Significance threshold", f"{FDR_ALPHA}"),
    ("Config used", "Significance column used", f"{SIGNIFICANCE_COL} (see the p-value glossary "
                                                 "entry above for why this isn't the FDR-corrected "
                                                 "column right now)"),
    ("Config used", "Minimum years for consistency ranking", f"{MIN_YEARS_FOR_CONSISTENCY}"),
    ("Config used", "Robustness minimum (Shortlist)", f"{ROBUSTNESS_MIN} ({int(ROBUSTNESS_MIN*100)}% of days profitable)"),
    ("Config used", "Profit Factor minimum (Shortlist)", f"{PROFIT_FACTOR_MIN}"),
    ("Config used", "Expectancy minimum (Shortlist)", f"${EXPECTANCY_MIN} per trade"),
    ("Config used", "IQR low-outlier fence multiplier", f"{IQR_K}"),
    ("Config used", "Top N (Winners/Losers)", f"{TOP_N}"),
]
guide_tab = pd.DataFrame(guide_rows, columns=["Section", "Item", "Explanation"])


# %% [CELL 13] ===============================================================
# WRITE THE ANALYSIS WORKBOOK — Combo-Day Detail is intentionally NOT
# included (it lives only in the Parquet master cache under MASTER_DIR);
# at this scale it would blow past Excel's ~1.05M row limit for no
# analytical benefit. Every sheet gets: a filterable, coloured header row,
# a frozen header, and auto-fit column widths -- except "Combo Parameters",
# which is kept narrow (it's the long raw combo-folder string) so it
# doesn't blow out the sheet; widen it manually or double-click the column
# border to see it in full.
# =============================================================================
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

HEADER_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
HEADER_FONT = Font(color="FFFFFF", bold=True)
NARROW_COLS = {"Combo Parameters"}
NARROW_WIDTH = 16
MAX_WIDTH = 32


def style_sheet(writer, sheet_name, df):
    ws = writer.sheets[sheet_name]
    if df.empty:
        return
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    dup_cols = df.columns[df.columns.duplicated()].tolist()
    if dup_cols:
        print(f"WARNING: sheet '{sheet_name}' has duplicate column name(s) {dup_cols} -- "
              f"only the first is styled/sized correctly; consider renaming upstream.")
    for idx, col in enumerate(df.columns, start=1):
        cell = ws.cell(row=1, column=idx)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        col_letter = get_column_letter(idx)
        if col in NARROW_COLS:
            ws.column_dimensions[col_letter].width = NARROW_WIDTH
        else:
            series = df.iloc[:, idx - 1]  # positional -- safe even if `col` label is duplicated
            # .astype(str) leaves NaN as an actual float instead of stringifying it (a general
            # pandas quirk, not just for category dtype), so len() on it crashes -- map explicitly
            # instead, treating missing values as an empty string for width purposes.
            sample = series.head(1000).map(lambda v: "" if pd.isna(v) else str(v))
            max_len = max([len(str(col))] + [len(v) for v in sample])
            ws.column_dimensions[col_letter].width = min(max(10, max_len + 2), MAX_WIDTH)


SHEETS = [
    ("Guide", guide_tab),
    ("Ranking", ranking_tab),
    ("Top Winners", top_winners),
    ("Top Losers", top_losers),
    ("Top Winners (Significant)", top_winners_sig),
    ("Top Losers (Significant)", top_losers_sig),
    ("Shortlist", shortlist),
    ("Parameter Performance", param_perf),
    ("Best Parameters", param_best),
    ("Worst Parameters", param_worst),
    ("Parameter Sensitivity", param_sensitivity),
    ("Total", total_tab),
    ("Yearly", yearly_tab),
    ("Quarterly", quarterly_tab),
    ("Symbol Stats", symbol_stats),
]

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    for sheet_name, df in SHEETS:
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        style_sheet(writer, sheet_name, df)

print(f"Workbook written: {OUT_XLSX}")
print(f"Master Parquet cache: {MASTER_DIR} (combo_days.parquet, symbol_days.parquet)")

try:
    from google.colab import files
    files.download(OUT_XLSX)
except ImportError:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config loaded.
Scanning 12 folder(s) under /content/drive/MyDrive/Colab/Alpaca_Experiment_Runs:
  RUNS_Orb_15_30_Multiparameters/2023: 12 zip(s)
  RUNS_Orb_15_30_Multiparameters/2024: 12 zip(s)
  RUNS_Orb_15_30_Multiparameters/2025: 12 zip(s)
  RUNS_Orb_10_20_MultiParameters/2023: 12 zip(s)
  RUNS_Orb_10_20_MultiParameters/2024: 12 zip(s)
  RUNS_Orb_10_20_MultiParameters/2025: 12 zip(s)
  RUNS_Orb_10_20_MultiParameters/2024_V2: 4 zip(s)
  RUNS_Orb_10_20_MultiParameters/2025_V2: 4 zip(s)
  RUNS_Orb5_MultiParameters/2023: 12 zip(s)
  RUNS_Orb5_MultiParameters/2024: 12 zip(s)
  RUNS_Orb5_MultiParameters/2025: 6 zip(s)
  RUNS_Orb5_MultiParameters/2026: 5 zip(s)
Parsing 115 zip(s) with 2 worker process(es)...
  [cached] results_Orbs_15_30_Apr2023.zip: 12096 combo-day rows, 226 symbol-day rows
  [cached] results_Orbs_15_30_Dec_2023.zip: 12096 combo-day rows, 207 sy

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>